# Jour 1 · Baseline, prédictions et métriques

> **Version corrigée** — les cellules de solution sont signalées.

## Objectifs

- entraîner une baseline et un premier classifieur
- lire une matrice de confusion
- interpréter précision, rappel, F1 et seuil de décision


## Toujours commencer par une baseline

Une **baseline** est une règle volontairement simple qui fixe le niveau minimum à battre. Ici, `DummyClassifier` prédit selon une stratégie naïve. Si un modèle complexe ne fait pas mieux, sa complexité n'est pas justifiée.


Une baseline peut toujours prédire la classe majoritaire, tirer selon la fréquence des classes ou appliquer une petite règle métier. Elle ne cherche pas à être excellente : elle donne un point de comparaison compréhensible.

Dans le code, `DummyClassifier(strategy="most_frequent")` répond toujours avec la classe la plus fréquente. La régression logistique doit apporter une information utile au-delà de cette stratégie naïve.


![Étapes fit et predict](../../ressources/illustrations/jour_01/03_fit_predict.png)

*fit apprend sur le train ; predict utilise ensuite les paramètres appris sur de nouvelles lignes.*


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_course_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "ressources" / "donnees" / "maintenance_machines.csv").exists():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable. Ouvrez le notebook depuis le dossier du cours.")


ROOT = find_course_root()
DATA_FILE = ROOT / "ressources" / "donnees" / "maintenance_machines.csv"
plt.style.use("seaborn-v0_8-whitegrid")

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

df = pd.read_csv(DATA_FILE)
features = ["temperature_c", "vibration_mm_s", "pressure_bar", "load_pct", "age_years", "days_since_maintenance"]
X = df[features]
y = df["failure_7d"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]
print("Baseline accuracy :", round(accuracy_score(y_test, baseline_pred), 3))
print("Modèle accuracy   :", round(accuracy_score(y_test, pred), 3))


## Pourquoi l'accuracy peut tromper

Avec peu de pannes, prédire toujours « normal » produit une accuracy élevée. Il faut regarder les quatre issues : vrai positif, faux positif, vrai négatif et faux négatif.


![Exemple d’accuracy élevée mais rappel nul](../../ressources/illustrations/jour_01/03_accuracy_trompeuse.png)

L’**accuracy** est la proportion totale de bonnes réponses. Elle traite toutefois toutes les lignes de la même façon. Si 94 machines sur 100 ne tombent pas en panne, répondre toujours « normal » donne 94 % d’accuracy tout en manquant les six pannes.

Une métrique doit donc être reliée à la question métier. Pour une classe rare, on regarde séparément la capacité à détecter les événements et la quantité de fausses alertes produites.


![Matrice de confusion en quatre cases](../../ressources/illustrations/jour_01/03_matrice_confusion.png)

*La matrice de confusion distingue les bonnes décisions des deux types d’erreurs.*


### Les quatre cases, en langage métier

- **Vrai positif** : une panne est annoncée et elle survient.
- **Faux positif** : une alerte est produite mais aucune panne ne survient.
- **Vrai négatif** : la machine reste normale et le modèle n’alerte pas.
- **Faux négatif** : aucune alerte n’est produite alors qu’une panne survient.

Le mot *positif* signifie ici « le modèle ou la réalité indique la classe panne ». Il ne signifie pas « bon résultat ».

Les deux erreurs n’ont pas nécessairement le même coût. Un faux positif peut déclencher une inspection inutile ; un faux négatif peut laisser survenir une panne non anticipée. Cette asymétrie explique pourquoi le choix d’une métrique et d’un seuil est une décision métier.


In [ ]:
cm = confusion_matrix(y_test, pred)
cm_df = pd.DataFrame(cm, index=["réel normal", "réel panne"], columns=["prédit normal", "prédit panne"])
cm_df


## Précision, rappel et F1

La **précision** répond : parmi toutes les alertes produites, quelle proportion correspondait réellement à une panne ? Le **rappel** répond : parmi toutes les pannes réelles, quelle proportion a été détectée ? Le **F1** combine précision et rappel et devient élevé seulement si les deux restent convenables.


![Questions posées par la précision et le rappel](../../ressources/illustrations/jour_01/03_precision_rappel.png)

*Une équipe qui ne tolère pas les pannes manquées privilégiera souvent davantage le rappel.*


### Un exemple chiffré

Supposons que le modèle produise 20 alertes et que 8 correspondent réellement à une panne. Sa précision vaut `8 / 20`, soit 40 %. S’il existait 10 pannes au total et qu’il en a trouvé 8, son rappel vaut `8 / 10`, soit 80 %.

Le **score F1** résume précision et rappel par une moyenne qui pénalise fortement une valeur très faible. Il est utile pour comparer, mais il masque le nombre réel d’alertes et ne connaît pas le coût métier. On conserve donc toujours les métriques détaillées.


In [ ]:
metrics = pd.Series({
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "recall": recall_score(y_test, pred, zero_division=0),
    "f1": f1_score(y_test, pred, zero_division=0),
    "alertes": int(pred.sum()),
})
metrics.round(3)


## Une probabilité devient une décision grâce à un seuil

`predict_proba` produit ici un score entre 0 et 1. Par défaut, un seuil proche de 0,5 transforme ce score en classe. Baisser le seuil crée généralement davantage d'alertes et augmente le rappel, mais peut réduire la précision.


![Effet du seuil de décision](../../ressources/illustrations/jour_01/03_seuil_decision.png)

Dans `predict_proba(X_test)[:, 1]`, `[:, 1]` sélectionne pour chaque ligne le score associé à la classe `1`, ici la panne. Un score de 0,70 signifie que le modèle favorise fortement cette classe selon ce qu’il a appris ; ce n’est pas une promesse individuelle de panne.

Si le seuil baisse, davantage de machines passent du côté « alerte » : on trouve souvent plus de pannes, mais on crée aussi davantage de faux positifs. S’il monte, le nombre d’alertes diminue et certaines pannes peuvent être manquées.

Dans un projet réel, on choisit ce compromis sur la validation. Le test reste fermé jusqu’à la fin. L’exercice utilise ici les mêmes lignes uniquement pour rendre l’effet des seuils visible ; il ne faut pas réutiliser le test pour optimiser un système final.


## À vous de jouer — comparer trois seuils

Comparez les seuils 0,30, 0,50 et 0,70. Affichez pour chacun le nombre d’alertes, la précision et le rappel.


In [ ]:
rows = []
for threshold in [0.30, 0.50, 0.70]:
    flags = (proba >= threshold).astype(int)
    rows.append({
        "seuil": threshold,
        "alertes": int(flags.sum()),
        "precision": precision_score(y_test, flags, zero_division=0),
        "recall": recall_score(y_test, flags, zero_division=0),
    })
pd.DataFrame(rows).set_index("seuil").round(3)


**À retenir :** le seuil encode un compromis opérationnel ; il n'existe pas de valeur universellement correcte.
